In [1]:
import random

import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import load_img, img_to_array
import numpy as np
import sklearn
from sklearn.model_selection import train_test_split
import pandas as pd

In [18]:
#load an preprocess data

def dataset_to_images_and_labels(dataset):
    images = []
    labels = []

    # Iterate through the dataset
    for image_batch, label_batch in dataset:
        images.append(image_batch.numpy())  # Convert Tensor to numpy array and store
        labels.append(label_batch.numpy())

    # Concatenate lists into numpy arrays
    images = np.concatenate(images, axis=0)
    labels = np.concatenate(labels, axis=0)
    return images, labels

def invert_colors(image, label):
    # Invert colors by subtracting from 255
    image = 255 - image
    return image, label

def random_transform(row):
    # Generate a random offset and scale factor
    offset_down = np.random.uniform(0, 100)
    transformed_row = row - offset_down
    transformed_row = np.clip(transformed_row, 0, 255)
    offset_up = np.random.uniform(0, offset_down)
    transformed_row = row - offset_down
    transformed_row = np.clip(transformed_row, 0, 255)

    scale = np.random.uniform(0.1, 1)
    transformed_row = transformed_row * scale

    return transformed_row

batch_size = 32
img_height = 28
img_width = 28

# Define path to dataset directory
training_set = pd.read_csv('28x28 dataset/mnist_train.csv')
test_set = pd.read_csv('28x28 dataset/mnist_test.csv')

training_set_labels = training_set['label']
test_set_labels = test_set['label']

training_set_pixel_data = training_set.drop(columns=['label'])
test_set_pixel_data = test_set.drop(columns=['label'])

#reshaping pixel_data  # Reshape to (num_samples, height, width, channels)
training_set_pixel_data = training_set_pixel_data.values.reshape(-1, img_height, img_width, 1)
test_set_pixel_data = test_set_pixel_data.values.reshape(-1, img_height, img_width, 1)

# creating tensors
training_images = tf.convert_to_tensor(training_set_pixel_data.tolist(), dtype=tf.int32)
training_labels = tf.convert_to_tensor(training_set_labels.tolist(), dtype=tf.int32)

testing_images = tf.convert_to_tensor(test_set_pixel_data.tolist(), dtype=tf.int32)
testing_labels = tf.convert_to_tensor(test_set_labels.tolist(), dtype=tf.int32)

# Create TensorFlow datasets
training_dataset = tf.data.Dataset.from_tensor_slices((training_images, training_labels))
training_dataset = training_dataset.shuffle(buffer_size=len(training_images)).batch(batch_size).prefetch(tf.data.AUTOTUNE)

testing_dataset = tf.data.Dataset.from_tensor_slices((testing_images, testing_labels))
testing_dataset = testing_dataset.shuffle(buffer_size=len(testing_images)).batch(batch_size).prefetch(tf.data.AUTOTUNE)

#erweitern von training dataset
inverted_training_dataset = training_dataset.map(invert_colors)
training_dataset = training_dataset.concatenate(inverted_training_dataset)

#
# """
# for images, labels in dataset.take(1):  # Take one batch from the dataset
#     print("Image batch shape:", images.shape)
#     print("Label batch shape:", labels.shape)
#     print("Image data type:", images.dtype)
#     print("Label data type:", labels.dtype)
#     print("First image in batch:", images[0].numpy())
#     print("First label in batch:", labels[0].numpy())
# """
#
# Initialize empty lists to hold images and labels
training_images, training_labels = dataset_to_images_and_labels(training_dataset)
testing_images, testing_labels = dataset_to_images_and_labels(testing_dataset)

# #save numpy Arrays
# np.save('images2.npy', images)
# np.save('labels2.npy', labels)

In [22]:
#Train
from tensorflow.keras.callbacks import EarlyStopping


X_train, X_validate, y_train, y_validate = train_test_split(training_images, training_labels, test_size=0.2, random_state=42)


# Define the model................................................................................................
model = models.Sequential([
    layers.Input(shape=(img_width, img_height, 1)),
    layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
    layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
    layers.MaxPooling2D((2, 2)),
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(10, activation='softmax'),
])

model.summary()

optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)

model.compile(
    optimizer=optimizer,                # Optimizer
    loss='sparse_categorical_crossentropy',  # Loss function for integer labels
    metrics=['accuracy'],       # Metrics to monitor
)

early_stop = EarlyStopping(
    monitor='val_loss',          # Val Loss beobachten
    patience=3,                  # 3 Epochs warten bevor Stopp
    restore_best_weights=True    # Gewichte vom besten Epoch wiederherstellen
)

history = model.fit(X_train, y_train, epochs=10, validation_data=(X_validate, y_validate),callbacks=early_stop, verbose=2)
model.save('digit_recognition_modelMinst2.h5')  # Save the model

Model: "sequential_9"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d_18 (Conv2D)          (None, 28, 28, 32)        320       
                                                                 
 conv2d_19 (Conv2D)          (None, 28, 28, 64)        18496     
                                                                 
 max_pooling2d_5 (MaxPooling  (None, 14, 14, 64)       0         
 2D)                                                             
                                                                 
 flatten_9 (Flatten)         (None, 12544)             0         
                                                                 
 dense_14 (Dense)            (None, 128)               1605760   
                                                                 
 dropout (Dropout)           (None, 128)               0         
                                                      

In [23]:
#Evaluate
loaded_model = tf.keras.models.load_model('digit_recognition_modelMinst2.h5')
test_loss, test_acc = loaded_model.evaluate(testing_images, testing_labels)
print(f'Test accuracy: {test_acc}')

313/313 [==============================] - 2s 5ms/step - loss: 0.0576 - accuracy: 0.9842
Test accuracy: 0.9842000007629395


In [26]:
#Testen mit meinen Bildern

print("Meine Bilder: ")
meineBilderDir = 'MeineTestBilder28x28'

dataset = tf.keras.preprocessing.image_dataset_from_directory(
    meineBilderDir,
    image_size=(img_height, img_width),
    color_mode='grayscale',  # since digits are usually grayscale
    batch_size=batch_size,
    label_mode='int'  # assumes labels are subdirectory names
)

# Initialize empty lists to hold images and labels
myImages = []
myLabels = []

# Iterate through the dataset
for image_batch, label_batch in dataset:
    myImages.append(image_batch.numpy())  # Convert Tensor to numpy array and store
    myLabels.append(label_batch.numpy())

# Concatenate lists into numpy arrays
myImages = np.concatenate(myImages, axis=0)
myLabels = np.concatenate(myLabels, axis=0)


predictions = loaded_model.predict(myImages)
print("Meine Bilder Predictions, Answer")
for i in range(len(predictions)):
    print(np.argmax(predictions[i]), myLabels[i])


# print("test Single Image")
# single_image_path = 'MeineTestBilder28x28/9/img.jpg'
# single_image = tf.keras.preprocessing.image.load_img(single_image_path, target_size=(img_height, img_width), color_mode="grayscale")
# single_image_array = tf.keras.preprocessing.image.img_to_array(single_image)
# single_image_array = np.expand_dims(single_image_array, axis=0)
# singlePredict = loaded_model.predict(single_image_array)
#
# print(np.argmax(singlePredict), singlePredict)




Meine Bilder: 
Found 7 files belonging to 10 classes.
1/1 [==============================] - 0s 26ms/step
Meine Bilder Predictions, Answer
7 7
9 9
3 3
5 6
5 5
2 2
9 8
